# Demo 05 : Permissions and execution

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · permissions

Write rules are matched in order. A write under /reports/ is allowed, and every other write is denied. A read with no matching rule proceeds.

In [2]:
import json
import sqlite3
import threading
import time
from pathlib import Path

from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain.messages import AIMessage, HumanMessage, ToolMessage
from langchain.tools import ToolRuntime, tool
from langchain_core.callbacks import BaseCallbackHandler
from deepagents import create_deep_agent
from deepagents.backends import StateBackend, FilesystemBackend, StoreBackend, CompositeBackend
from deepagents.backends.utils import create_file_data
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


class Counter(BaseCallbackHandler):
    def __init__(self):
        self.lock = threading.Lock()
        self.reset()

    def reset(self):
        self.calls = self.input = self.output = self.invalid = self.errors = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        with self.lock:
            self.calls += 1

    def on_llm_end(self, response, **kwargs):
        with self.lock:
            for generations in response.generations:
                for generation in generations:
                    message = getattr(generation, "message", None)
                    usage = getattr(message, "usage_metadata", None) or {}
                    self.input += usage.get("input_tokens", 0)
                    self.output += usage.get("output_tokens", 0)
                    self.invalid += len(getattr(message, "invalid_tool_calls", []) or [])

    def on_llm_error(self, error, **kwargs):
        with self.lock:
            self.errors += 1


counter = Counter()
llm = make_llm(callbacks=[counter], max_tokens=1800, max_retries=0)
metrics = []
LIMIT = {"recursion_limit": 24}
DB = Path(data("demo-world/service.db")).resolve()
MANUALS = Path(data_folder("demo-world/manuals"))
PAGES = {p.name: p.read_text(encoding="utf-8") for p in sorted(MANUALS.glob("*.md"))}
WORKSPACE = (Path.cwd() / "workspace").resolve()
WORKSPACE.mkdir(exist_ok=True)


def ask(text):
    return {"messages": [HumanMessage(text)]}


def answer(result):
    return str(next(m.content for m in reversed(result["messages"])
                    if isinstance(m, AIMessage) and m.content and not m.tool_calls))


def measured(label, operation):
    counter.reset()
    start = time.perf_counter()
    try:
        return operation()
    finally:
        row = dict(label=label, calls=counter.calls, input_tokens=counter.input,
                   output_tokens=counter.output, malformed=counter.invalid, errors=counter.errors,
                   seconds=round(time.perf_counter() - start, 1))
        metrics.append(row)
        print("metrics :", json.dumps(row))


async def measured_async(label, operation):
    counter.reset()
    start = time.perf_counter()
    try:
        return await operation()
    finally:
        row = dict(label=label, calls=counter.calls, input_tokens=counter.input,
                   output_tokens=counter.output, malformed=counter.invalid, errors=counter.errors,
                   seconds=round(time.perf_counter() - start, 1))
        metrics.append(row)
        print("metrics :", json.dumps(row))


def show(result):
    print("tools :", [m.name for m in result["messages"] if m.type == "tool"])
    print("answer :", answer(result))


@tool
def part_stock(part: str) -> str:
    """Look up stock, storage location and name for a Veenhaven Bus part number."""
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        rows = db.execute("SELECT part_no, name, stock, depot FROM parts WHERE part_no = ?",
                          (part.upper(),)).fetchall()
    return json.dumps(rows)


@tool
def fault_guidance(error_code: str) -> str:
    """Find the manual's meaning, first action and replacement condition for a bus error code."""
    return "\n".join(f"[{name}]\n{line}" for name, text in PAGES.items()
                     for line in text.splitlines()
                     if line.startswith("| " + error_code.upper() + " |")) or "Unknown error code."


STOCK_TASK = "Look up CE-CF-02 stock and location. Answer briefly from the tool."
MANUAL_TASK = "Look up F-CE-110. Give the first action and replacement condition, citing the file."
QUESTION = ("Bus CE12-2021-0117 shows F-CE-110. How many CE-CF-02 cooling fans are in stock, "
            "at which location, and what are the manual's first action and replacement condition? Be brief.")

In [3]:
from deepagents import FilesystemPermission

rules = [
    FilesystemPermission(operations=["write"], paths=["/reports/**"], mode="allow"),
    FilesystemPermission(operations=["write"], paths=["/**"], mode="deny"),
]
gated = create_deep_agent(model=llm, permissions=rules, system_prompt=(
    "Use only write_file. Write /scratch.md with the single word fan. "
    "If that call is denied, write /reports/fan.md with the single word fan. Then stop."))
result = measured("permissions", lambda: gated.invoke(ask(
    "Write the note, using the fallback path if the first write is denied."), LIMIT))
show(result)
for message in result["messages"]:
    if message.type == "tool":
        print(message.name, ":", str(message.content)[:240])
print("files :", sorted(result.get("files", {})))

metrics : {"label": "permissions", "calls": 3, "input_tokens": 6197, "output_tokens": 58, "malformed": 0, "errors": 0, "seconds": 8.5}
tools : ['write_file', 'write_file']
answer : Done.
write_file : Error: permission denied for write on /scratch.md
write_file : Updated file /reports/fan.md
files : ['/reports/fan.md']


## b · the-interpreter

We attach the QuickJS interpreter and ask the model to write JavaScript that totals ticket hours. A read-only tool loads the rows into the REPL through programmatic tool calling. The rows stay in the environment; only the calculation comes back to the model. We print the generated code and check its result against Python.

In [4]:
import csv
import importlib.metadata as metadata
import xml.etree.ElementTree as ET
from decimal import Decimal
from deepagents.backends.protocol import SandboxBackendProtocol
from langchain_quickjs import CodeInterpreterMiddleware, PTCOption

with Path(data("demo-world/tickets.csv")).open(encoding="utf-8", newline="") as handle:
    tickets = list(csv.DictReader(handle))

@tool
def ticket_rows() -> str:
    """Load all Veenhaven Bus tickets as JSON rows with depot_id, hours and parts_used."""
    return json.dumps(tickets)

ptc: PTCOption = [ticket_rows]
interpreter = CodeInterpreterMiddleware(ptc=ptc, subagents=False, mode="turn",
    memory_limit=16 * 1024 * 1024, timeout=2, max_ptc_calls=2, max_result_chars=1200)
analyst = create_deep_agent(model=llm, middleware=[interpreter], system_prompt=(
    "Use only eval. Load ticket data inside JavaScript with JSON.parse(await tools.ticketRows({})). "
    "Compute from those rows; do not return or print the raw rows. Return the requested JSON "
    "using JSON.stringify as the last expression. Then answer briefly from the result."))
print("versions :", metadata.version("deepagents"), metadata.version("langchain-quickjs"))
print("tools :", sorted(analyst.get_graph().nodes["tools"].data.tools_by_name))
result = await measured_async("quickjs-tickets", lambda: analyst.ainvoke(ask(
    "For depot D01, count tickets and sum hours only where parts_used is empty. "
    "Return JSON with count and hours. In the same JSON add host APIs with "
    "typeof process, typeof require and typeof fetch. Use one eval call."), LIMIT))
for message in result["messages"]:
    for call in getattr(message, "tool_calls", []):
        if call["name"] == "eval":
            print("JavaScript :\n" + call["args"]["code"])
    if message.type == "tool" and message.name == "eval":
        print("sandbox :", message.content)
        computed = json.loads(ET.fromstring(message.content).text)
show(result)
selected = [row for row in tickets if row["depot_id"] == "D01" and not row["parts_used"]]
expected = {"count": len(selected), "hours": float(sum(Decimal(row["hours"]) for row in selected))}
print("Python check :", expected)
assert {key: computed[key] for key in expected} == expected

versions : 0.7.30 0.3.9
tools : ['delete', 'edit_file', 'eval', 'execute', 'glob', 'grep', 'ls', 'read_file', 'task', 'write_file']


metrics : {"label": "quickjs-tickets", "calls": 2, "input_tokens": 5662, "output_tokens": 163, "malformed": 0, "errors": 0, "seconds": 5.7}
JavaScript :
const rows = JSON.parse(await tools.ticketRows({})); const filtered = rows.filter(r => r.depot_id === "D01" && (r.parts_used == null || r.parts_used === "" || (Array.isArray(r.parts_used) && r.parts_used.length === 0))); const result = {count: filtered.length, hours: filtered.reduce((sum, r) => sum + Number(r.hours || 0), 0), hostAPIs: {process: typeof process, require: typeof require, fetch: typeof fetch}}; JSON.stringify(result);
sandbox : <result>{"count":4,"hours":7.5,"hostAPIs":{"process":"undefined","require":"undefined","fetch":"undefined"}}</result>
tools : ['eval']
answer : {"count":4,"hours":7.5,"hostAPIs":{"process":"undefined","require":"undefined","fetch":"undefined"}}
Python check : {'count': 4, 'hours': 7.5}


We expose only the ticket loader to JavaScript. `eval` has no built-in host filesystem, network, process or clock API. An exposed Python tool still runs with the host's permissions. The VM timeout covers JavaScript execution; waiting for a host tool can take longer. `execute` remains a separate shell tool, and a state backend does not make that shell a sandbox.

## c · a-local-shell

LocalShellBackend runs the host shell with the workspace folder as its working directory. virtual_mode confines file tools to that folder and leaves execute unrestricted, so the command is one echo and the process environment is not inherited. The tool result is the command output ; the model's retelling can change the spacing.

In [5]:
from deepagents.backends import LocalShellBackend

SHELL = WORKSPACE / "shell"
SHELL.mkdir(exist_ok=True)
local = LocalShellBackend(root_dir=SHELL, virtual_mode=True, inherit_env=False)
print("cwd :", local.cwd)
print("implements execute protocol :", isinstance(local, SandboxBackendProtocol))
shell_agent = create_deep_agent(model=llm, backend=local, system_prompt=(
    "Call execute once with exactly this command: echo veenhaven. "
    "Report the output and stop. Do not use another tool."))
result = measured("local-shell", lambda: shell_agent.invoke(ask("Run the echo."), LIMIT))
show(result)
for message in result["messages"]:
    if message.type == "tool":
        print(message.name, ":", str(message.content)[:300])

cwd : C:\Git\courseWare\Multi-Agents-LangGraph\notebooks\08-deep-agents\workspace\shell
implements execute protocol : True


metrics : {"label": "local-shell", "calls": 2, "input_tokens": 4549, "output_tokens": 31, "malformed": 0, "errors": 0, "seconds": 4.2}
tools : ['execute']
answer : veenhaven
execute : veenhaven

[Command succeeded with exit code 0]


## d · an-override-and-its-rules

create_deep_agent copies permissions onto its own FilesystemMiddleware. A FilesystemMiddleware passed in middleware replaces that instance by name, and on this pin the replacement starts with no rules. A scripted model makes the same read_file call three times, so no course model is called : with the harness middleware, with a plain override, and with an override that receives the rules itself.

In [6]:
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from deepagents.middleware.filesystem import FilesystemMiddleware

class Scripted(GenericFakeChatModel):
    def bind_tools(self, tools, **kwargs):
        return self

def scripted_read(path):
    return Scripted(messages=iter([
        AIMessage("", tool_calls=[{"name": "read_file", "args": {"file_path": path}, "id": "read-1"}]),
        AIMessage("done")]))

deny_secret = [FilesystemPermission(operations=["read"], paths=["/secret/**"], mode="deny")]
seeded = {"/secret/export.md": create_file_data("SECRET EXPORT")}
backend = StateBackend()
arms = {
    "harness middleware": dict(),
    "plain override": dict(backend=backend, middleware=[FilesystemMiddleware(backend=backend)]),
    "override with _permissions": dict(backend=backend, middleware=[
        FilesystemMiddleware(backend=backend, _permissions=deny_secret)]),
}
for label, extra in arms.items():
    agent = create_deep_agent(model=scripted_read("/secret/export.md"), permissions=deny_secret, **extra)
    result = agent.invoke({**ask("Read the export."), "files": seeded})
    print(f"{label:28}:", [str(m.content).replace("\n", " ") for m in result["messages"] if m.type == "tool"])

harness middleware          : ['Error: permission denied for read on /secret/export.md']
plain override              : ['@@ lines 1-1 of 1 @@ SECRET EXPORT']
override with _permissions  : ['Error: permission denied for read on /secret/export.md']


`_permissions` is a private argument. The release after this pin, 0.7.31, copies the rules onto a replacement, and it needs langchain-core 1.7, so this room stays on 0.7.30. A symlink check is in the pin : since 0.7.25 a rule is matched against the requested path and against its symlink-resolved target.

## Things to try

Optional : we can change the bus task and compare the answer, calls and tokens.